# GAT — Graph Attention Networks (2017)

_Papers · Graph Neural Networks_

**Let each node LEARN how much to listen to each neighbor, instead of using fixed graph weights.**

---

This notebook is a practice scaffold for the **GAT — Graph Attention Networks (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example (Eqns 3-4). W = I so Wh = h. ---
h_ex = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])      # rows: h_i, h_j, h_k  (N_i = {i,j,k})
a_ex = torch.tensor([0.2, 0.3, -0.6, 0.5])          # a = [a_src || a_dst], length 2F'
leaky = nn.LeakyReLU(0.2)
Wh_i = h_ex[0]
raw = torch.stack([a_ex[:2] @ Wh_i + a_ex[2:] @ h_ex[n] for n in range(3)])  # a^T[Wh_i || Wh_n]
e   = leaky(raw)
alpha = torch.softmax(e, dim=0)
h_i_new = (alpha.unsqueeze(1) * h_ex).sum(0)        # sum_j alpha_ij * Wh_j
print("raw   :", [round(v,4) for v in raw.tolist()])      # [-0.4, 0.7, 0.1]
print("e     :", [round(v,4) for v in e.tolist()])        # [-0.08, 0.7, 0.1]
print("alpha :", [round(v,4) for v in alpha.tolist()])    # [0.2284, 0.4982, 0.2734]
print("h_i'  :", [round(v,4) for v in h_i_new.tolist()])  # [0.5018, 0.7716]


# --- 1. The graph attention layer (built by hand). learned=False -> GCN-style ablation. ---
class GATLayer(nn.Module):
    def __init__(self, in_f, out_f, learned=True):
        super().__init__()
        self.learned = learned
        self.W = nn.Linear(in_f, out_f, bias=False)
        self.a_src = nn.Parameter(torch.empty(out_f)); nn.init.normal_(self.a_src, std=0.3)
        self.a_dst = nn.Parameter(torch.empty(out_f)); nn.init.normal_(self.a_dst, std=0.3)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, h, adj):
        Wh = self.W(h)                                  # (N, out_f)   the shared transform
        N = Wh.size(0)
        if self.learned:
            s_src = (Wh * self.a_src).sum(-1, keepdim=True)   # a_src^T Wh_i  -> (N,1)
            s_dst = (Wh * self.a_dst).sum(-1, keepdim=True)   # a_dst^T Wh_j  -> (N,1)
            e = self.leaky(s_src + s_dst.T)                   # (N,N): score for edge i<-j (Eqn. 1)
            e = e.masked_fill(adj == 0, float("-inf"))        # MASK non-edges (Eqn. 3)
            alpha = torch.softmax(e, dim=1)                   # softmax over neighbors -> rows sum to 1
        else:
            # GCN-style fixed weights 1/sqrt(d_i d_j) over the (self-looped) adjacency.
            deg = adj.sum(1)
            norm = adj / (deg.sqrt().unsqueeze(1) * deg.sqrt().unsqueeze(0))
            alpha = norm
        return alpha @ Wh, alpha                              # h' = sum_j alpha_ij Wh_j  (Eqn. 4)


class GAT(nn.Module):
    def __init__(self, in_f, hid, n_cls, heads=4, learned=True):
        super().__init__()
        self.heads = nn.ModuleList([GATLayer(in_f, hid, learned) for _ in range(heads)])  # head 1
        self.out   = GATLayer(hid * heads, n_cls, learned)                                # output layer
        self.elu   = nn.ELU()

    def forward(self, h, adj):
        outs = [head(h, adj)[0] for head in self.heads]
        h1 = self.elu(torch.cat(outs, dim=1))          # CONCAT heads on the hidden layer (Eqn. 5)
        logits, alpha = self.out(h1, adj)              # output layer (single head here)
        return logits, alpha


# --- 2. A tiny toy graph: 8 nodes, two communities; node features carry the signal. ---
#  Nodes 0-3 = class 0, nodes 4-7 = class 1. Edges within and a couple across communities.
N = 8
edges = [(0,1),(0,2),(1,3),(2,3),(4,5),(4,6),(5,7),(6,7),(3,4)]   # 3-4 bridges the communities
adj = torch.eye(N)                                    # self-loops: i in N_i
for u, v in edges:
    adj[u, v] = 1; adj[v, u] = 1
y = torch.tensor([0,0,0,0,1,1,1,1])
g = torch.Generator().manual_seed(1)
centers = torch.tensor([[2.,0.],[0.,2.]])             # class-0 vs class-1 feature centers
h = centers[y] + 0.5 * torch.randn(N, 2, generator=g)


def train(learned, epochs=120):
    torch.manual_seed(0)
    net = GAT(2, 4, 2, heads=4, learned=learned)
    opt = torch.optim.Adam(net.parameters(), lr=0.05, weight_decay=5e-4)
    lf  = nn.CrossEntropyLoss()
    for ep in range(epochs):
        opt.zero_grad(); logits, _ = net(h, adj); loss = lf(logits, y)
        loss.backward(); opt.step()
    logits, alpha = net(h, adj)
    acc = (logits.argmax(1) == y).float().mean().item()
    return acc, alpha, loss.item()

acc_gat, alpha_gat, loss_gat = train(learned=True)
acc_gcn, alpha_gcn, _        = train(learned=False)   # ABLATION: fixed 1/sqrt(d_i d_j)
print(f"\nGAT  (learned alpha): train acc {acc_gat:.3f}   final loss {loss_gat:.4f}")
print(f"GCN  (fixed   alpha): train acc {acc_gcn:.3f}")

# Read out node 4's learned attention. Node 4 is class 1; its neighbors are node 3 (class 0,
# the cross-class bridge), itself, and nodes 5,6 (class 1).
nbrs = (adj[4] > 0).nonzero().squeeze(1).tolist()
print("\nNode 4 neighbors (class):", [(j, y[j].item()) for j in nbrs])  # [(3,0),(4,1),(5,1),(6,1)]
print("GAT learned alpha[4]:", [round(alpha_gat[4, j].item(), 3) for j in nbrs])  # ~[0.017,0.024,0.413,0.546]
print("GCN fixed   alpha[4]:", [round(alpha_gcn[4, j].item(), 3) for j in nbrs])  # ~[0.25, 0.25, 0.289, 0.289]
# GAT LEARNS to nearly ignore node 3 (the cross-class bridge, weight ~0.017) and lean on its
# same-class neighbors; GCN hands node 3 a fixed ~0.25 it cannot lower.
# (Exact numbers vary by seed/hardware; this is our small run, not the paper's reported result.)

## Visualize the data & results

_At a node with a cross-class neighbor, does GAT's LEARNED attention down-weight that misleading neighbor, where GCN's FIXED weights cannot?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Reproduce the qualitative effect: learned attention down-weights a cross-class
# neighbor where GCN's fixed degree weights cannot. Toy 8-node graph.
torch.manual_seed(0)
N = 8
edges = [(0,1),(0,2),(1,3),(2,3),(4,5),(4,6),(5,7),(6,7),(3,4)]
adj = torch.eye(N)
for u,v in edges: adj[u,v]=1; adj[v,u]=1
y = torch.tensor([0,0,0,0,1,1,1,1])
g = torch.Generator().manual_seed(1)
centers = torch.tensor([[2.,0.],[0.,2.]])
h = centers[y] + 0.5*torch.randn(N,2,generator=g)

class GATLayer(nn.Module):
    def __init__(self, i, o, learned=True):
        super().__init__(); self.learned=learned
        self.W=nn.Linear(i,o,bias=False)
        self.a_src=nn.Parameter(torch.randn(o)*0.3); self.a_dst=nn.Parameter(torch.randn(o)*0.3)
        self.leaky=nn.LeakyReLU(0.2)
    def forward(self, h, adj):
        Wh=self.W(h)
        if self.learned:
            e=self.leaky((Wh*self.a_src).sum(-1,keepdim=True)+(Wh*self.a_dst).sum(-1,keepdim=True).T)
            e=e.masked_fill(adj==0,float("-inf")); alpha=torch.softmax(e,1)
        else:
            d=adj.sum(1); alpha=adj/(d.sqrt().unsqueeze(1)*d.sqrt().unsqueeze(0))
        return alpha@Wh, alpha

class GAT(nn.Module):
    def __init__(self, learned=True, heads=4):
        super().__init__()
        self.heads=nn.ModuleList([GATLayer(2,4,learned) for _ in range(heads)])
        self.out=GATLayer(4*heads,2,learned); self.elu=nn.ELU()
    def forward(self,h,adj):
        h1=self.elu(torch.cat([hd(h,adj)[0] for hd in self.heads],1))
        return self.out(h1,adj)

def run(learned):
    torch.manual_seed(0); net=GAT(learned)
    opt=torch.optim.Adam(net.parameters(),lr=0.05,weight_decay=5e-4); lf=nn.CrossEntropyLoss()
    for _ in range(120):
        opt.zero_grad(); lo,_=net(h,adj); lf(lo,y).backward(); opt.step()
    return net(h,adj)[1]

ag = run(True); ac = run(False)
nbrs=(adj[4]>0).nonzero().squeeze(1).tolist()
print("neighbors of node 4 (class):", [(j,y[j].item()) for j in nbrs])  # [(3,0),(4,1),(5,1),(6,1)]
print("GAT  alpha[4]:", [round(ag[4,j].item(),3) for j in nbrs])  # ~[0.017,0.024,0.413,0.546]
print("GCN  alpha[4]:", [round(ac[4,j].item(),3) for j in nbrs])  # ~[0.25,0.25,0.289,0.289]
# GAT nearly ignores node 3 (cross-class bridge, ~0.017); GCN gives it a fixed ~0.25.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a trained tiny GAT whose attention $\alpha$ is skewed toward the
            informative neighbor. Replace the learned $\alpha_{ij}$ with GCN's fixed
            $1/\sqrt{d_i d_j}$ (everything else identical) and retrain. What changes, and what does it show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap only the weighting: instead of softmax-over-LeakyReLU scores, hard-code each neighbor's weight to $1/\sqrt{d_i d_j}$ from the (self-looped) adjacency. Keep $W$, the data, optimizer, and seed fixed. — _An honest ablation changes exactly one thing &mdash; learned vs fixed edge weights &mdash; so any difference is attributable to attention._
- Retrain and inspect: the GCN-style version weights every same-degree neighbor equally and cannot down-weight the noisy one, so on a task where one neighbor is misleading it should reach higher loss / lower accuracy than GAT. — _The fixed rule ignores features; GAT's learned $\alpha$ can route around the noisy neighbor._
- Conclude that the gain comes from learning the per-neighbor weights, not from the message-passing structure (which both share). — _Both models aggregate over the same neighborhood with the same $W$; only the weighting differs, isolating attention as the cause._

**Answer:** The fixed-weight (GCN-style) layer must give equal say to same-degree neighbors, so when one
                 neighbor is noisy it drags the node's representation; GAT's learned $\alpha$ down-weights that
                 neighbor and keeps lower loss / higher accuracy. Since the only difference is learned vs fixed
                 edge weights, this isolates attention as the source of the gain &mdash; exactly the
                 GAT-vs-GCN contrast of &sect;2.2. The CODEVIZ panel shows the learned weights skewing toward
                 the informative neighbor.

</details>

**Problem 2.** Recompute one attention coefficient. For the worked example
            ($W=I$, $\vec{h}_i=[1,0]$, $\vec{h}_k=[1,1]$, $\vec{a}=[0.2,0.3\|-0.6,0.5]$), find the raw
            score $e_{ik}$ and explain where LeakyReLU would matter.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Score: $\vec{a}^{\top}[W\vec{h}_i \| W\vec{h}_k] = 0.2(1)+0.3(0) + (-0.6)(1)+0.5(1) = 0.2 - 0.6 + 0.5 = 0.10$. — _First two terms read node $i$ ($0.2$); last two read neighbor $k$ ($-0.1$)._
- LeakyReLU$(0.10)=0.10$ (positive, passes through unchanged). — _For a positive score LeakyReLU is the identity; the $0.2$ slope only changes negatives._
- Contrast the self-edge $e_{ii}$: its raw score was $-0.40$, so LeakyReLU gives $0.2\times(-0.40)=-0.08$ &mdash; a plain ReLU would have made it $0$ and dropped its gradient. — _That is precisely why the paper uses LeakyReLU on the score._

**Answer:** $e_{ik}=\text{LeakyReLU}(0.10)=0.10$. LeakyReLU is the identity here because the score is
                 positive; it only bites on negative scores like the self-edge's $-0.40\to-0.08$, where a plain
                 ReLU would zero it out and kill the gradient. This matches the notebook's printed
                 $e_{ik}=0.10$.

</details>

**Problem 3.** Your tiny graph has a node $i$ with degree 4 and a node $m$ with degree 1, and you stack two GAT
            layers. After training, node $i$'s four attention weights come out as $[0.05, 0.80, 0.10, 0.05]$.
            What does that tell you, and how would GCN have weighted the same four neighbors?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the GAT weights: they sum to ~1 and are heavily skewed to the second neighbor (0.80). — _$\alpha$ is a softmax over neighbors, so it is a distribution; a peak means the layer learned that neighbor is most relevant._
- Compare to GCN: it would assign each neighbor $1/\sqrt{d_i d_j}$ &mdash; a value fixed by degrees, identical for any two neighbors of the same degree, independent of features. — _GCN's rule cannot peak on a feature-relevant neighbor; it only reflects graph structure._
- Conclude GAT is using feature content, not just wiring, to route information. — _The skew is impossible under the degree-only rule, so it must come from the learned, feature-dependent score._

**Answer:** The skewed weights $[0.05,0.80,0.10,0.05]$ mean GAT learned that the second neighbor
                 is by far the most informative and routed most of the aggregation through it. GCN would instead
                 hand each neighbor the fixed $1/\sqrt{d_i d_j}$ &mdash; equal for any same-degree neighbors and
                 blind to their features &mdash; so it could never produce that peak. This is the core GAT
                 advantage from &sect;2.2: learned, feature-dependent edge weights.

</details>